# Notebook 02: Feature Extraction -- ChangeMyView (CMV)

Extracts GoEmotions distributions and computes rolling AMD trajectories
from Reddit ChangeMyView threads via the ConvoKit corpus.

Each "conversation" is a dyadic exchange: the OP and one top-level
respondent's reply chain. Delta-awarded threads indicate persuasion
success (meaning convergence); no-delta threads serve as controls.

Outputs:
- `cmv_features.parquet`: per-utterance features with GoEmotions distributions
- `cmv_conversation_summary.parquet`: per-thread AMD trajectory and delta label

In [ ]:
!pip install -q convokit transformers torch pandas numpy tqdm scikit-learn pyarrow nltk

In [ ]:
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load CMV Corpus via ConvoKit

In [ ]:
from convokit import Corpus, download

corpus = Corpus(filename=download("winning-args-corpus"))
print(f"Utterances: {len(list(corpus.iter_utterances()))}")
print(f"Conversations: {len(list(corpus.iter_conversations()))}")
print(f"Speakers: {len(list(corpus.iter_speakers()))}")

In [ ]:
MIN_TURNS = 6
MAX_TURNS = 100

conversations = []
skipped_short = 0
skipped_mono = 0

for convo in corpus.iter_conversations():
    utts = sorted(convo.iter_utterances(), key=lambda u: u.timestamp if u.timestamp else 0)

    if len(utts) < 2:
        continue

    op_speaker = utts[0].speaker.id

    pair_threads = defaultdict(lambda: {"success": None, "utt_ids": set()})

    for u in utts:
        meta = u.meta if u.meta else {}
        pair_ids = meta.get("pair_ids", [])
        success = meta.get("success", None)

        if success is None or pair_ids is None:
            continue

        for pid in pair_ids:
            pair_threads[pid]["success"] = int(success)
            pair_threads[pid]["utt_ids"].add(u.id)

    utt_by_id = {u.id: u for u in utts}

    def build_reply_chain(start_utt_id):
        chain = []
        visited = set()
        queue = [start_utt_id]
        while queue:
            uid = queue.pop(0)
            if uid in visited or uid not in utt_by_id:
                continue
            visited.add(uid)
            u = utt_by_id[uid]
            chain.append(u)
            replies = u.meta.get("replies", []) if u.meta else []
            if replies:
                queue.extend(replies)
        chain.sort(key=lambda u: u.timestamp if u.timestamp else 0)
        return chain

    seen_threads = set()

    for pid, info in pair_threads.items():
        success_val = info["success"]
        if success_val is None:
            continue

        thread_key = frozenset(info["utt_ids"])
        if thread_key in seen_threads:
            continue
        seen_threads.add(thread_key)

        thread_utts_raw = [utt_by_id[uid] for uid in info["utt_ids"] if uid in utt_by_id]
        thread_utts_raw.sort(key=lambda u: u.timestamp if u.timestamp else 0)

        speakers_in = set(u.speaker.id for u in thread_utts_raw)
        if op_speaker not in speakers_in:
            thread_utts_raw = [utt_by_id[utts[0].id]] + thread_utts_raw
            speakers_in.add(op_speaker)

        respondent_counts = defaultdict(int)
        for u in thread_utts_raw:
            if u.speaker.id != op_speaker:
                respondent_counts[u.speaker.id] += 1

        if not respondent_counts:
            skipped_mono += 1
            continue

        best_respondent = max(respondent_counts.keys(), key=lambda s: respondent_counts[s])

        thread_utts = []
        for u in thread_utts_raw:
            if u.speaker.id not in (op_speaker, best_respondent):
                continue
            text = u.text.strip() if u.text else ""
            if len(text) < 5:
                continue
            thread_utts.append({
                "speaker": u.speaker.id,
                "text": text[:1024],
                "timestamp": u.timestamp,
                "utt_id": u.id,
            })

        if len(thread_utts) < MIN_TURNS:
            skipped_short += 1
            continue
        thread_utts = thread_utts[:MAX_TURNS]

        final_speakers = set(u["speaker"] for u in thread_utts)
        if len(final_speakers) < 2:
            skipped_mono += 1
            continue

        conversations.append({
            "convo_id": f"{convo.id}__{pid}",
            "delta": bool(success_val == 1),
            "op_speaker": op_speaker,
            "respondent": best_respondent,
            "utterances": thread_utts,
        })

n_delta = sum(1 for c in conversations if c["delta"])
n_nodelta = len(conversations) - n_delta
print(f"Reply threads extracted: {len(conversations)}")
print(f"  Delta (successful): {n_delta}")
print(f"  No-delta (unsuccessful): {n_nodelta}")
print(f"  Skipped (too short): {skipped_short}")
print(f"  Skipped (mono-speaker): {skipped_mono}")
if conversations:
    print(f"  Min turns: {min(len(c['utterances']) for c in conversations)}")
    print(f"  Median turns: {np.median([len(c['utterances']) for c in conversations]):.0f}")
    print(f"  Max turns: {max(len(c['utterances']) for c in conversations)}")

## 2. Extract GoEmotions Distributions

In [ ]:
GOEMOTIONS_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral",
]

goemotions_pipeline = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    device=0 if DEVICE == "cuda" else -1,
    truncation=True,
    max_length=512,
)
print("GoEmotions pipeline loaded")

In [ ]:
BATCH_SIZE = 64

all_texts = []
text_index = []
for ci, convo in enumerate(conversations):
    for ui, utt in enumerate(convo["utterances"]):
        all_texts.append(utt["text"])
        text_index.append((ci, ui))

print(f"Total texts to process: {len(all_texts)}")

all_dists = [None] * len(all_texts)

for batch_start in tqdm(range(0, len(all_texts), BATCH_SIZE), desc="GoEmotions"):
    batch_texts = all_texts[batch_start:batch_start + BATCH_SIZE]
    batch_results = goemotions_pipeline(batch_texts)

    for j, result in enumerate(batch_results):
        label_scores = {item["label"]: item["score"] for item in result}
        dist = [label_scores.get(emo, 0.0) for emo in GOEMOTIONS_LABELS]
        total = sum(dist)
        if total > 0:
            dist = [d / total for d in dist]
        all_dists[batch_start + j] = dist

for (ci, ui), dist in zip(text_index, all_dists):
    conversations[ci]["utterances"][ui]["goemotions_dist"] = dist

print(f"GoEmotions extraction complete: {len(all_dists)} utterances")

## 3. Topic Clustering (Context Variable)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

all_thread_texts = [u["text"] for c in conversations for u in c["utterances"]]

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english", min_df=2)
tfidf_matrix = vectorizer.fit_transform(all_thread_texts)

TOPIC_K = 5
kmeans = KMeans(n_clusters=TOPIC_K, random_state=RANDOM_SEED, n_init=10)
cluster_labels = kmeans.fit_predict(tfidf_matrix)

idx = 0
for convo in conversations:
    for utt in convo["utterances"]:
        utt["topic_cluster"] = int(cluster_labels[idx])
        utt["context"] = f"c{cluster_labels[idx]}"
        idx += 1

print(f"Topic clustering complete: {TOPIC_K} clusters")
print(f"Cluster distribution: {Counter(cluster_labels).most_common()}")

## 4. Compute Rolling AMD Trajectories

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))
WORD_PATTERN = re.compile(r"\b[a-z]{2,}\b")
MIN_ANCHOR_FREQ = 2
MIN_PER_CELL = 1
ROLLING_WINDOW = 5
N_EMO = len(GOEMOTIONS_LABELS)


def extract_content_words(text):
    words = WORD_PATTERN.findall(text.lower())
    return [w for w in words if w not in STOPWORDS]


def total_variation(p, q):
    return 0.5 * np.sum(np.abs(np.asarray(p) - np.asarray(q)))


def compute_rolling_amd(records, spk1, spk2, window_size):
    n = len(records)
    if n < window_size:
        return [], []

    d_marg_series = []
    d_cond_series = []

    for start in range(n - window_size + 1):
        window = records[start:start + window_size]

        words_s1, words_s2 = [], []
        for r in window:
            cw = extract_content_words(r["text"])
            if r["speaker"] == spk1:
                words_s1.extend(cw)
            else:
                words_s2.extend(cw)

        c1, c2 = Counter(words_s1), Counter(words_s2)
        shared = set(c1.keys()) & set(c2.keys())
        anchors = [w for w in shared if c1[w] + c2[w] >= MIN_ANCHOR_FREQ]

        if not anchors:
            d_marg_series.append(np.nan)
            d_cond_series.append(np.nan)
            continue

        anchor_map = defaultdict(lambda: defaultdict(list))
        for r in window:
            text_words = set(WORD_PATTERN.findall(r["text"].lower()))
            for anchor in anchors:
                if anchor in text_words:
                    anchor_map[(anchor, r["speaker"])][r["context"]].append(
                        np.array(r["goemotions_dist"])
                    )

        d_marg_vals, d_cond_vals = [], []

        for anchor in anchors:
            g1 = anchor_map.get((anchor, spk1), {})
            g2 = anchor_map.get((anchor, spk2), {})

            ed1, ed2, cnt1, cnt2 = {}, {}, {}, {}
            for ctx, dists in g1.items():
                if len(dists) >= MIN_PER_CELL:
                    ed1[ctx] = np.mean(dists, axis=0)
                    cnt1[ctx] = len(dists)
            for ctx, dists in g2.items():
                if len(dists) >= MIN_PER_CELL:
                    ed2[ctx] = np.mean(dists, axis=0)
                    cnt2[ctx] = len(dists)

            if not ed1 or not ed2:
                continue

            t1 = sum(cnt1.values())
            t2 = sum(cnt2.values())
            cw1 = {c: n / t1 for c, n in cnt1.items()}
            cw2 = {c: n / t2 for c, n in cnt2.items()}

            all_ctx = set(cw1.keys()) | set(cw2.keys())
            marg1 = sum(cw1.get(c, 0) * ed1.get(c, np.zeros(N_EMO)) for c in all_ctx)
            marg2 = sum(cw2.get(c, 0) * ed2.get(c, np.zeros(N_EMO)) for c in all_ctx)
            d_marg_vals.append(total_variation(marg1, marg2))

            shared_ctx = set(ed1.keys()) & set(ed2.keys())
            if shared_ctx:
                wt_sum, tv_sum = 0.0, 0.0
                for c in shared_ctx:
                    w = cnt1[c] + cnt2[c]
                    tv_sum += w * total_variation(ed1[c], ed2[c])
                    wt_sum += w
                if wt_sum > 0:
                    d_cond_vals.append(tv_sum / wt_sum)

        d_marg_series.append(np.mean(d_marg_vals) if d_marg_vals else np.nan)
        d_cond_series.append(np.mean(d_cond_vals) if d_cond_vals else np.nan)

    return d_marg_series, d_cond_series


print(f"Rolling AMD config: window={ROLLING_WINDOW}, "
      f"min_anchor_freq={MIN_ANCHOR_FREQ}, min_per_cell={MIN_PER_CELL}")

In [ ]:
from scipy.stats import kendalltau

ROLLING_CSD = 5

def rolling_lag1_ac(series, w):
    series = np.asarray(series, dtype=float)
    n_out = len(series) - w + 1
    if n_out <= 0:
        return np.array([])
    result = np.full(n_out, np.nan)
    for i in range(n_out):
        win = series[i:i + w]
        if len(win) < 3:
            continue
        x, y = win[:-1], win[1:]
        mx, my = np.mean(x), np.mean(y)
        denom = np.sqrt(np.sum((x - mx)**2) * np.sum((y - my)**2))
        result[i] = np.sum((x - mx) * (y - my)) / denom if denom > 0 else 0.0
    return result

def rolling_var(series, w):
    series = np.asarray(series, dtype=float)
    n_out = len(series) - w + 1
    if n_out <= 0:
        return np.array([])
    result = np.full(n_out, np.nan)
    for i in range(n_out):
        win = series[i:i + w]
        result[i] = np.var(win, ddof=1) if len(win) > 1 else 0.0
    return result

conversation_summaries = []

for convo in tqdm(conversations, desc="Computing rolling AMD + CSD"):
    records = convo["utterances"]
    spk1 = convo["op_speaker"]
    spk2 = convo["respondent"]

    d_marg_series, d_cond_series = compute_rolling_amd(
        records, spk1, spk2, ROLLING_WINDOW
    )

    d_marg_arr = np.array(d_marg_series, dtype=float)
    d_cond_arr = np.array(d_cond_series, dtype=float)

    valid_marg = d_marg_arr[~np.isnan(d_marg_arr)]
    valid_cond = d_cond_arr[~np.isnan(d_cond_arr)]

    if len(valid_cond) >= 3:
        tau_cond, p_tau_cond = kendalltau(np.arange(len(valid_cond)), valid_cond)
    else:
        tau_cond, p_tau_cond = np.nan, np.nan

    if len(valid_marg) >= 3:
        tau_marg, p_tau_marg = kendalltau(np.arange(len(valid_marg)), valid_marg)
    else:
        tau_marg, p_tau_marg = np.nan, np.nan

    n_valid_cond = int((~np.isnan(d_cond_arr)).sum())

    first_half_cond = valid_cond[:len(valid_cond)//2] if len(valid_cond) >= 4 else np.array([])
    second_half_cond = valid_cond[len(valid_cond)//2:] if len(valid_cond) >= 4 else np.array([])

    mean_first = np.mean(first_half_cond) if len(first_half_cond) > 0 else np.nan
    mean_second = np.mean(second_half_cond) if len(second_half_cond) > 0 else np.nan
    d_cond_change = mean_second - mean_first if not (np.isnan(mean_first) or np.isnan(mean_second)) else np.nan

    word_counts = [len(r["text"].split()) for r in records]
    mean_wc = np.mean(word_counts)
    total_words = sum(word_counts)

    wc_s1 = [len(r["text"].split()) for r in records if r["speaker"] == spk1]
    wc_s2 = [len(r["text"].split()) for r in records if r["speaker"] == spk2]
    mean_wc_op = np.mean(wc_s1) if wc_s1 else 0.0
    mean_wc_resp = np.mean(wc_s2) if wc_s2 else 0.0

    ge_matrix = np.array([r["goemotions_dist"] for r in records])
    turn_ge_var = np.var(ge_matrix, axis=1)

    ac1_ge_var = rolling_lag1_ac(turn_ge_var, ROLLING_CSD)
    var_ge_var = rolling_var(turn_ge_var, ROLLING_CSD)
    valid_ac1 = ac1_ge_var[~np.isnan(ac1_ge_var)]
    valid_var = var_ge_var[~np.isnan(var_ge_var)]

    if len(valid_ac1) >= 3:
        tau_ac1, _ = kendalltau(np.arange(len(valid_ac1)), valid_ac1)
    else:
        tau_ac1 = np.nan
    if len(valid_var) >= 3:
        tau_var, _ = kendalltau(np.arange(len(valid_var)), valid_var)
    else:
        tau_var = np.nan

    if len(valid_cond) >= ROLLING_CSD:
        ac1_dcond = rolling_lag1_ac(valid_cond, ROLLING_CSD)
        var_dcond = rolling_var(valid_cond, ROLLING_CSD)
        v_ac1_dc = ac1_dcond[~np.isnan(ac1_dcond)]
        v_var_dc = var_dcond[~np.isnan(var_dcond)]
        tau_ac1_dcond = kendalltau(np.arange(len(v_ac1_dc)), v_ac1_dc)[0] if len(v_ac1_dc) >= 3 else np.nan
        tau_var_dcond = kendalltau(np.arange(len(v_var_dc)), v_var_dc)[0] if len(v_var_dc) >= 3 else np.nan
    else:
        tau_ac1_dcond, tau_var_dcond = np.nan, np.nan

    conversation_summaries.append({
        "convo_id": convo["convo_id"],
        "delta": convo["delta"],
        "n_turns": len(records),
        "n_rolling_points": len(d_marg_series),
        "n_valid_cond": n_valid_cond,
        "mean_d_marg": np.nanmean(d_marg_arr) if len(valid_marg) > 0 else np.nan,
        "mean_d_cond": np.nanmean(d_cond_arr) if len(valid_cond) > 0 else np.nan,
        "tau_d_cond": tau_cond,
        "p_tau_d_cond": p_tau_cond,
        "tau_d_marg": tau_marg,
        "d_cond_first_half": mean_first,
        "d_cond_second_half": mean_second,
        "d_cond_change": d_cond_change,
        "mean_word_count": mean_wc,
        "total_words": total_words,
        "mean_wc_op": mean_wc_op,
        "mean_wc_resp": mean_wc_resp,
        "tau_ac1_ge_var": tau_ac1,
        "tau_var_ge_var": tau_var,
        "tau_ac1_dcond": tau_ac1_dcond,
        "tau_var_dcond": tau_var_dcond,
        "d_marg_series": d_marg_series,
        "d_cond_series": d_cond_series,
    })

summary_df = pd.DataFrame(conversation_summaries)
valid = summary_df.dropna(subset=["tau_d_cond"])
print(f"Total conversations: {len(summary_df)}")
print(f"With valid D_cond trajectory: {len(valid)}")
print(f"  Delta: {valid['delta'].sum()}, No-delta: {(~valid['delta']).sum()}")
print(f"\nD_cond Kendall tau by outcome:")
for label, grp in valid.groupby("delta"):
    tag = "delta" if label else "no-delta"
    print(f"  {tag}: mean tau = {grp['tau_d_cond'].mean():.4f}, "
          f"median = {grp['tau_d_cond'].median():.4f}")
print(f"\nEngagement stats:")
print(f"  Mean word count per turn: {summary_df['mean_word_count'].mean():.1f}")
print(f"  Mean total words: {summary_df['total_words'].mean():.0f}")
print(f"\nCSD indicators (GoEmotions variance):")
for label, grp in valid.groupby("delta"):
    tag = "delta" if label else "no-delta"
    print(f"  {tag}: AC1 tau = {grp['tau_ac1_ge_var'].mean():.4f}, "
          f"Var tau = {grp['tau_var_ge_var'].mean():.4f}")

## 5. Save Features to Parquet

In [ ]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

flat_records = []
for convo in conversations:
    for ui, utt in enumerate(convo["utterances"]):
        flat = {
            "convo_id": convo["convo_id"],
            "delta": convo["delta"],
            "turn_idx": ui,
            "speaker": utt["speaker"],
            "is_op": utt["speaker"] == convo["op_speaker"],
            "text": utt["text"],
            "topic_cluster": utt["topic_cluster"],
            "context": utt["context"],
        }
        for emo_idx, emo_name in enumerate(GOEMOTIONS_LABELS):
            flat[f"ge_{emo_name}"] = utt["goemotions_dist"][emo_idx]
        flat_records.append(flat)

utt_df = pd.DataFrame(flat_records)
utt_df.to_parquet(OUTPUT_DIR / "cmv_features.parquet", index=False)
print(f"Utterance features saved: {utt_df.shape}")

drop_cols = ["d_marg_series", "d_cond_series"]
summary_save = summary_df.drop(columns=[c for c in drop_cols if c in summary_df.columns])
summary_save.to_parquet(OUTPUT_DIR / "cmv_conversation_summary.parquet", index=False)
print(f"Conversation summaries saved: {summary_save.shape}")
print(f"Columns: {list(summary_save.columns)}")

print(f"\nSummary stats:")
print(summary_save[["mean_d_marg", "mean_d_cond", "tau_d_cond", "d_cond_change",
                     "mean_word_count", "tau_ac1_ge_var", "tau_var_ge_var"]].describe())

## [Colab only] Save outputs to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil

DRIVE_DEST = Path("/content/drive/MyDrive/phase-transition-amd/data")
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

for f in ["cmv_features.parquet", "cmv_conversation_summary.parquet"]:
    src = OUTPUT_DIR / f
    if src.exists():
        shutil.copy2(src, DRIVE_DEST / f)
        print(f"Copied {f} -> {DRIVE_DEST}")

print(f"\nAll CMV outputs saved to Google Drive at: {DRIVE_DEST}")